# **Ansible**

## **Playbooks**

#### vars_dir_creation.yaml

---
- name: My First Playbook
  hosts: #localhost
  connection: #local
  vars:
    dir_path: #/home/labex/project/test_directory
    file_content: #"Hello from Ansible! The time is {{ ansible_date_time.iso8601 }}"

  tasks:
    - name: Create a directory
      file:
        path: "{{ dir_path }}"
        state: directory
        mode: "0755"

    - name: Create a file
      copy:
        content: "{{ file_content }}"
        dest: "{{ dir_path }}/hello.txt"

    - name: Display file content
      debug:
        msg: "The content of the file is: {{ file_content }}"

#### test-config.yaml

---
- name: Test Custom Configuration
  hosts: all
  tasks:
    - name: Display remote user
      debug:
        msg: "Connected as user: {{ ansible_user }}"

    - name: Display privilege escalation info
      debug:
        msg: "Privilege escalation is {{ 'enabled' if ansible_become | default(false) else 'disabled' }}"

    - name: Show Ansible configuration
      debug:
        msg: "Inventory file: {{ lookup('config', 'DEFAULT_HOST_LIST') }}"

    - name: Check if become is enabled in ansible.cfg
      command: grep "become = True" /home/labex/project/ansible.cfg
      register: become_check
      changed_when: false
      failed_when: false

    - name: Display become setting from ansible.cfg
      debug:
        msg: "Become is {{ 'enabled' if become_check.rc == 0 else 'disabled' }} in ansible.cfg"

#### host_info.yaml

---
- name: Gather system information from all hosts
  hosts: all
  gather_facts: true

  tasks:
    - name: Display system information
      ansible.builtin.debug:
        msg:
          - "Hostname: {{ ansible_facts['hostname'] }}"
          - "FQDN: {{ ansible_facts['fqdn'] }}"
          - "OS: {{ ansible_facts['distribution'] }} {{ ansible_facts['distribution_version'] }}"
          - "Kernel: {{ ansible_facts['kernel'] }}"
          - "Architecture: {{ ansible_facts['architecture'] }}"
          - "CPU: {{ ansible_facts['processor_vcpus'] }} vCPU(s)"
          - "Memory: {{ ansible_facts['memtotal_mb'] }} MB"
          - "Default IPv4: {{ ansible_facts['default_ipv4']['address'] | default('N/A') }}"
          - "Python: {{ ansible_facts['python_version'] }}"
          - "Uptime: {{ (ansible_facts['uptime_seconds'] / 86400) | int }} day(s)"

---
- name: Test connectivity to all hosts
  hosts: all
  gather_facts: false

  tasks:
    - name: Ping hosts
      ansible.builtin.ping:

#### sudoaccess.yaml

---
- name: Test sudo access on all hosts
  hosts: all
  gather_facts: true 
  become: true

  tasks:
    - name: Check current user
      ansible.builtin.command:
        cmd: whoami
      register: whoami_result
      changed_when: false

    - name: Display current user
      ansible.builtin.debug:
        msg: "Running as: {{ whoami_result.stdout }}"

## **Running Playbooks**

#### host Info

In [1]:
!ansible-playbook /home/sysadmin/Documents/ansible/host_info.yaml -i /home/sysadmin/Documents/ansible/inventory.ini


PLAY [Gather system information from all hosts] ********************************

TASK [Gathering Facts] *********************************************************
ok: [localhost]
ok: [k8s-controller]

TASK [Display system information] **********************************************
ok: [localhost] => {
    "msg": [
        "Hostname: cachyv2",
        "FQDN: cachyv2",
        "OS: Archlinux rolling",
        "Kernel: 7.1.5-1-cachyos",
        "Architecture: x86_64",
        "CPU: 16 vCPU(s)",
        "Memory: 15647 MB",
        "Default IPv4: 10.0.0.193",
        "Python: 3.14.7",
        "Uptime: 11 day(s)"
    ]
}
ok: [k8s-controller] => {
    "msg": [
        "Hostname: k8s-controller",
        "FQDN: k8s-controller",
        "OS: OracleLinux 9.8",
        "Kernel: 5.15.0-321.202.5.3.el9uek.x86_64",
        "Architecture: x86_64",
        "CPU: 2 vCPU(s)",
        "Memory: 7475 MB",
        "Default IPv4: 10.0.0.140",
        "Python: 3.9.25",
        "Uptime: 35 day(s)"
    ]
}

PL

#### ping Test

In [21]:
!ansible-playbook /home/sysadmin/Documents/ansible/ping.yaml -i /home/sysadmin/Documents/ansible/inventory.ini 


PLAY [Test connectivity to all hosts] ******************************************

TASK [Gathering Facts] *********************************************************
ok: [localhost]
ok: [k8s-controller]

TASK [Ping hosts] **************************************************************
ok: [localhost]
ok: [k8s-controller]

PLAY RECAP *********************************************************************
k8s-controller             : ok=2    changed=0    unreachable=0    failed=0    skipped=0    rescued=0    ignored=0   
localhost                  : ok=2    changed=0    unreachable=0    failed=0    skipped=0    rescued=0    ignored=0   



#### sudo Access

In [20]:
!ansible-playbook /home/sysadmin/Documents/ansible/test_sudo.yaml -i /home/sysadmin/Documents/ansible/inventory.ini 


PLAY [Test sudo access on all hosts] *******************************************

TASK [Gathering Facts] *********************************************************
ok: [localhost]
ok: [k8s-controller]

TASK [Check current user] ******************************************************
ok: [k8s-controller]
ok: [localhost]

TASK [Display current user] ****************************************************
ok: [localhost] => {
    "msg": "Running as: root"
}
ok: [k8s-controller] => {
    "msg": "Running as: root"
}

PLAY RECAP *********************************************************************
k8s-controller             : ok=3    changed=0    unreachable=0    failed=0    skipped=0    rescued=0    ignored=0   
localhost                  : ok=3    changed=0    unreachable=0    failed=0    skipped=0    rescued=0    ignored=0   



In [14]:
!ls -l /home/sysadmin/Documents/ansible/

total 16
-rw-r--r-- 1 sysadmin sysadmin 1938 Jul 20 23:58 create_and_switch_branch.yml
-rw-r--r-- 1 sysadmin sysadmin 1035 Aug 22 16:27 inventory.ini
-rw-r--r-- 1 sysadmin sysadmin  154 Aug 22 16:37 ping.yml
-rw-r--r-- 1 sysadmin sysadmin  357 Aug 22 16:40 test_sudo.yml


## **Command Onelines**

#### setup

In [ ]:
#This module when ran gathers facts about the remote hosts and returns them to the control node The gathered facts can be used in playbooks and templates to make decisions based on the state of the remote hosts
!ansible all -i /home/sysadmin/Documents/ansible/inventory.ini -m setup

In [ ]:
# This command only gathers hosts facts related to the distribution of the operating system -- The filter option allows you to specify which facts to gather, in this case, only those related to the distribution of the operating system
!ansible all -i /home/sysadmin/Documents/ansible/inventory.ini -m setup -a "filter=ansible_distribution*"

k8s-controller | SUCCESS => {
    "ansible_facts": {
        "ansible_distribution": "OracleLinux",
        "ansible_distribution_file_parsed": true,
        "ansible_distribution_file_path": "/etc/oracle-release",
        "ansible_distribution_file_search_string": "Oracle Linux",
        "ansible_distribution_file_variety": "OracleLinux",
        "ansible_distribution_major_version": "9",
        "ansible_distribution_release": "NA",
        "ansible_distribution_version": "9.8"
    },
    "changed": false
}
localhost | SUCCESS => {
    "ansible_facts": {
        "ansible_distribution": "Archlinux",
        "ansible_distribution_file_path": "/etc/arch-release",
        "ansible_distribution_file_variety": "Archlinux",
        "ansible_distribution_major_version": "rolling",
        "ansible_distribution_release": "n/a",
        "ansible_distribution_version": "rolling"
    },
    "changed": false
}


#### ram usage

In [13]:
!ansible all -i /home/sysadmin/Documents/ansible/inventory.ini -m command -a "free -h"

localhost | CHANGED | rc=0 >>
               total        used        free      shared  buff/cache   available
Mem:            15Gi       9.6Gi       4.5Gi       1.2Gi       3.0Gi       5.7Gi
Swap:           15Gi         9Gi       5.3Gi
k8s-controller | CHANGED | rc=0 >>
               total        used        free      shared  buff/cache   available
Mem:           7.3Gi       3.6Gi       408Mi       339Mi       3.9Gi       3.7Gi
Swap:             0B          0B          0B


In [15]:
#because the command module is the default module, you can omit the -m command option and just use -a to specify the command to run
!ansible all -i /home/sysadmin/Documents/ansible/inventory.ini -a "free -h"

localhost | CHANGED | rc=0 >>
               total        used        free      shared  buff/cache   available
Mem:            15Gi       9.4Gi       4.5Gi       1.2Gi       3.1Gi       5.9Gi
Swap:           15Gi        10Gi       5.2Gi
k8s-controller | CHANGED | rc=0 >>
               total        used        free      shared  buff/cache   available
Mem:           7.3Gi       3.6Gi       414Mi       339Mi       3.9Gi       3.7Gi
Swap:             0B          0B          0B


#### inventory info

In [10]:
!cat /home/sysadmin/Documents/ansible/inventory.ini

# Replace the example names/IPs below with your hosts.

[local]
localhost ansible_connection=local ansible_python_interpreter=/usr/bin/python3.14

[debian]
# debian-01 ansible_host=192.168.1.10 ansible_user=your_user
k8s-controller ansible_host=10.0.0.140 ansible_connection=ssh ansible_user=sysadmin ansible_python_interpreter=/usr/bin/python3.9
# ansible debain -i inventory.ini -m ping --ask-become-pass
# ansible_user=sysadmin

#[arch]
# arch-01 ansible_host=192.168.1.20 ansible_user=your_user
#localhost ansible_host=localhost
#ansible_user=sysadmin python_interpreter=/usr/bin/python3

[rhel]
# rhel-01 ansible_host=192.168.1.30 ansible_user=your_user

[all:vars]
ansible_user=sysadmin
ansible_password=Passw0rd
ansible_become=true
ansible_become_method=sudo
ansible_become_password=Passw0rd
#ansible_connection=ssh
#ansible_user=sysadmin
#ansible_become=true
#ansible_become_method=sudo
#
# ansible all -i inventory.ini -m ping -k --ask-become-pass
#
# ansible_user: sysadmin
# ansible_passwo

#### disk space

In [9]:
!ansible all -i /home/sysadmin/Documents/ansible/inventory.ini -m command -a "df -h"

localhost | CHANGED | rc=0 >>
Filesystem      Size  Used Avail Use% Mounted on
/dev/nvme0n1p2  932G   76G  853G   9% /
devtmpfs        7.5G     0  7.5G   0% /dev
tmpfs           7.7G  422M  7.3G   6% /dev/shm
efivarfs        256K  181K   71K  72% /sys/firmware/efi/efivars
tmpfs           3.1G  3.9M  3.1G   1% /run
none            1.0M     0  1.0M   0% /run/credentials/systemd-journald.service
tmpfs           7.7G  179M  7.5G   3% /tmp
/dev/nvme0n1p2  932G   76G  853G   9% /root
/dev/nvme0n1p2  932G   76G  853G   9% /srv
/dev/nvme0n1p2  932G   76G  853G   9% /home
/dev/nvme0n1p2  932G   76G  853G   9% /var/tmp
/dev/nvme0n1p2  932G   76G  853G   9% /var/log
/dev/nvme0n1p2  932G   76G  853G   9% /var/cache
/dev/nvme0n1p1  511M  664K  511M   1% /boot/efi
overlay         932G   76G  853G   9% /var/lib/docker/rootfs/overlayfs/06d26c8b67144bb0760e6f9ea19075498255b55dbbb2a04ce76e6feaad212ac6
overlay         932G   76G  853G   9% /var/lib/docker/rootfs/overlayfs/ca8f7a60397f1b44625ccb7d76bb2b18

#### ansible config

In [ ]:
!ansible-config dump

In [8]:
!ansible-config dump | grep -i Action

ACTION_WARNINGS(default) = True
DEFAULT_ACTION_PLUGIN_PATH(default) = ['/home/sysadmin/.ansible/plugins/action', '/usr/share/ansible/plugins/action']
VALIDATE_ACTION_GROUP_METADATA(default) = True


#### uptime

In [6]:
!ansible all -i /home/sysadmin/Documents/ansible/inventory.ini -a "uptime"

localhost | CHANGED | rc=0 >>
 21:05:47 up 12 days,  1:45,  6 users,  load average: 2.61, 2.94, 2.92
k8s-controller | CHANGED | rc=0 >>
 21:05:47 up 36 days, 7 min,  2 users,  load average: 0.16, 0.12, 0.07


In [12]:
!ansible local -i /home/sysadmin/Documents/ansible/inventory.ini -a "uptime"

localhost | CHANGED | rc=0 >>
 11:27:00 up 12 days, 16:06,  5 users,  load average: 3.44, 2.83, 2.67


In [11]:
!ansible debian -i /home/sysadmin/Documents/ansible/inventory.ini -a "uptime"

k8s-controller | CHANGED | rc=0 >>
 11:26:37 up 36 days, 14:27,  2 users,  load average: 0.05, 0.10, 0.09


#### whoami

In [8]:
!ansible all -i /home/sysadmin/Documents/ansible/inventory.ini -m command -a "whoami" --become

k8s-controller | CHANGED | rc=0 >>
root
localhost | CHANGED | rc=0 >>
root


#### Ping Commands

In [13]:
!ansible all -i /home/sysadmin/Documents/ansible/inventory.ini -m ping

localhost | SUCCESS => {
    "changed": false,
    "ping": "pong"
}
k8s-controller | SUCCESS => {
    "changed": false,
    "ping": "pong"
}


In [4]:
!ansible all -i /home/sysadmin/Documents/ansible/inventory.ini -m ping -e "ansible_user=sysadmin ansible_password=YourPassword"

[WARNING]: Host 'localhost' is using the discovered Python interpreter at '/usr/bin/python3.14', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
localhost | SUCCESS => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python3.14"
    },
    "changed": false,
    "ping": "pong"
}
[WARNING]: Host 'k8s-controller' is using the discovered Python interpreter at '/usr/bin/python3.9', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
k8s-controller | SUCCESS => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python3.9"
    },
    "changed": false,
    "ping": "pong"
}


## **IP Info**

In [1]:
!ip -br addr

lo               UNKNOWN        127.0.0.1/8 ::1/128 
eno1             DOWN           
wlan0            UP             10.0.0.193/24 2601:140:8285:cce0::1544/128 2601:140:8285:cce0:6a33:c7c3:574:ae19/64 fe80::361d:d76a:aa3b:dad/64 
br-0081ab6aab24  DOWN           172.22.0.1/16 
br-28ada3f9755d  UP             172.21.0.1/16 fe80::384e:31ff:fe25:a937/64 
br-2c0d8dd69729  DOWN           172.24.0.1/16 
br-73cba769a40c  UP             10.0.4.1/24 fe80::308c:e4ff:fecb:d9e4/64 
br-795c1e05c843  UP             10.0.2.1/24 fdfe:611b:f35c::1/64 fe80::494:7fff:fe68:66e9/64 
br-c3869c809198  UP             10.0.5.1/24 fe80::4c7a:4bff:fee3:617b/64 
br-6c8d27d1410b  UP             10.0.3.1/24 fe80::d096:17ff:fea3:c966/64 
br-7666c4be89ce  UP             172.18.0.1/16 fe80::8093:c5ff:fef6:f61b/64 
br-c30a29e7f04b  UP             172.23.0.1/16 fe80::904c:7aff:fef2:a716/64 
br-efdf0e9d5fdd  UP             172.25.0.1/16 fe80::c0b9:8ff:fe89:c906/64 
docker0          UP             10.0.1.1/24 fe80::8ce6:5

## **Inventory**

[local]
localhost ansible_connection=local ansible_python_interpreter=/usr/bin/python3.14

[debian]
k8s-controller ansible_host=10.0.0.140 ansible_connection=ssh ansible_user=sysadmin ansible_python_interpreter=/usr/bin/python3.9
#ansible debain -i inventory.ini -m ping --ask-become-pass*

#[arch]
#arch-01 ansible_host=192.168.1.20 ansible_user=your_user*
#localhost ansible_host=localhost
#ansible_user=sysadmin python_interpreter=/usr/bin/python3

[rhel]
#rhel-01 ansible_host=192.168.1.30 ansible_user=your_user

[all:vars]
ansible_user=sysadmin
ansible_password=***
ansible_become=true
ansible_become_method=sudo
ansible_become_password=***
#ansible_connection=ssh
#ansible_user=sysadmin
#ansible_become=true
#ansible_become_method=sudo
#ansible all -i inventory.ini -m ping -k --ask-become-pass
#ansible_user: sysadmin
#ansible_password: "{{ vault_ansible_password }}"